# 実データ正規化 v3

In [1]:
import pandas as pd
import os
os.makedirs("csvv3", exist_ok=True)

## 1. 二次医療圏（市区町村→医療圏）と政令市コード正規化

In [2]:
iry = pd.read_csv("二次医療圏/二次医療圏.csv")
iry["都道府県"] = iry["都道府県"].str.strip()
df_iryoken = iry[iry["都道府県"] == "静岡県"].copy()
df_iryoken["二次医療圏"] = df_iryoken["二次医療圏"].str.strip()
df_iryoken["市区町村"] = df_iryoken["市区町村"].str.strip()

# 市区町村名 → 属性
muni_lookup = df_iryoken.set_index("市区町村")[["市区町村ID", "二次医療圏ID", "二次医療圏"]].to_dict("index")

# --- 政令市の市区町村コード正規化 ---
# 避難所=市単位(22100/22130), 人口・医療圏=区単位, 病院住所=新区名 と粒度が異なるため、
# 静岡市・浜松市は「市単位」に集約して人口配分を整合させる。
def normalize_muni(mid):
    if pd.isna(mid):
        return mid
    mid = int(mid)
    if mid in (22101, 22102, 22103):   # 静岡市 葵/駿河/清水区
        return 22100
    if 22131 <= mid <= 22137:          # 浜松市 中/東/西/南/北/浜北/天竜区
        return 22130
    return mid

# 市区町村ID → 医療圏（政令市の市コードも補完）
area2_by_muni = df_iryoken.drop_duplicates("市区町村ID").set_index("市区町村ID")[
    ["二次医療圏ID", "二次医療圏"]].to_dict("index")
area2_by_muni[22100] = {"二次医療圏ID": 2205, "二次医療圏": "静岡"}
area2_by_muni[22130] = {"二次医療圏ID": 2208, "二次医療圏": "西部"}
df_iryoken.head()

,都道府県,二次医療圏ID,二次医療圏,市区町村ID,市区町村
1008,静岡県,2201,賀茂,22219,下田市
1009,静岡県,2201,賀茂,22301,東伊豆町
1010,静岡県,2201,賀茂,22302,河津町
1011,静岡県,2201,賀茂,22304,南伊豆町
1012,静岡県,2201,賀茂,22305,松崎町


## 2. メッシュ人口（町丁字別）

In [3]:
df = pd.read_csv("小地域集計　（主な内容：基本単位区別，町丁・字別人口など）/h03_22_加工済み.csv", encoding="Shift-JIS")
df = df[~df["大字・町名"].isna()]
df["町丁字コード"] = pd.to_numeric(df["町丁字コード"], errors="coerce")
df["町丁字グループ"] = (df["町丁字コード"] // 1000) * 1000
# 各町丁字系統で最も詳細な階層のみ残す
df = df[df.groupby(["市区町村コード", "大字・町名"])["地域階層レベル"].transform("max") == df["地域階層レベル"]]
df_jinko = df.merge(df_iryoken[["市区町村ID", "二次医療圏ID", "二次医療圏"]],
                    left_on="市区町村コード", right_on="市区町村ID", how="left")
df_jinko["総数"] = pd.to_numeric(df_jinko["総数"], errors="coerce").fillna(0)
print("町丁字数:", len(df_jinko), " 合計人口:", f"{df_jinko['総数'].sum():,.0f}")

町丁字数: 4769  合計人口: 3,596,994


## 3. 避難所（共通IDに市区町村コード。政令市は市単位22100/22130）

In [4]:
df_h = pd.read_csv("避難所等データ/22000_1_指定避難所/22000_1.csv", encoding="utf-8")
df_h["市区町村ID"] = df_h["共通ID"].str[1:6].astype(int)
# 政令市の市コードも補完済みの area2_by_muni で医療圏を付与
df_h["二次医療圏ID"] = df_h["市区町村ID"].map(lambda m: area2_by_muni.get(m, {}).get("二次医療圏ID"))
df_h["二次医療圏"] = df_h["市区町村ID"].map(lambda m: area2_by_muni.get(m, {}).get("二次医療圏"))
df_hinanzyo = df_h
miss = df_hinanzyo[df_hinanzyo["二次医療圏ID"].isna()]
if len(miss):
    print("医療圏が付かない避難所:", len(miss), "市区町村ID:", sorted(miss["市区町村ID"].unique()))
print("避難所数:", len(df_hinanzyo))

避難所数: 1701


## 4. 病院

- 対象は `医療救護①`・`医療救護②`。プレースホルダ「―」を除外。


In [5]:
dfb = pd.read_csv("災害時における医療/08medcareindisasters_geocoded.csv")
dfb = dfb[dfb["section"].isin(["医療救護①", "医療救護②"])].copy()
dfb = dfb[dfb["hospital_name"].astype(str).str.strip() != "―"].copy()
dfb = dfb[dfb["address"].notna() & dfb["latitude"].notna() & dfb["longitude"].notna()].copy()
# ★物理的に一意化（医療救護①/②の重複や複数掲載を1件に）
dfb = dfb.drop_duplicates(subset=["hospital_name", "address"]).reset_index(drop=True)

# ★住所から市区町村IDを判定（政令市は市単位に割当）
muni_names = list(muni_lookup.keys())
def resolve_municipality_id(address):
    a = str(address)
    if "静岡市" in a: return 22100
    if "浜松市" in a: return 22130
    for name in sorted(muni_names, key=len, reverse=True):   # 長い名前優先（富士宮市>富士市）
        if name in a:
            return muni_lookup[name]["市区町村ID"]
    return None
dfb["市区町村ID"] = dfb["address"].apply(resolve_municipality_id)

# ★誤ジオコーディング病院の手動補正（住所切り詰め7件など。確定値を key:院名 で追記）
#   例) MANUAL_MUNI_FIX = {"公益社団法人有隣厚生会富士病院": 22210, ...}
MANUAL_MUNI_FIX = {
    "ＪＡ静岡厚生連中伊豆温泉病院": 22222,       # 伊豆市下白岩
    "伊豆保健医療センター": 22225,               # 伊豆の国市田京
    "瀬尾記念・慶友病院": 22203,                 # 沼津市下香貫
    "医療法人社団宏和会岡村記念病院": 22341,      # 駿東郡清水町柿田
    "公益社団法人有隣厚生会富士病院": 22215,      # 御殿場市ぐみ沢（名称に反し御殿場市）
    "公益社団法人有隣厚生会富士小山病院": 22344,  # 駿東郡小山町用沢
    "医療法人社団駿栄会御殿場石川病院": 22215,    # 御殿場市深沢
}
for name, mid in MANUAL_MUNI_FIX.items():
    dfb.loc[dfb["hospital_name"] == name, "市区町村ID"] = mid

miss = dfb[dfb["市区町村ID"].isna()]
if len(miss):
    print("【警告】市区町村を判定できない病院:")
    print(miss[["hospital_name", "address"]].to_string())

dfb["二次医療圏ID"] = dfb["市区町村ID"].map(lambda m: area2_by_muni.get(m, {}).get("二次医療圏ID"))
dfb["二次医療圏"] = dfb["市区町村ID"].map(lambda m: area2_by_muni.get(m, {}).get("二次医療圏"))
df_byoin = dfb
print("一意な病院数:", len(df_byoin), "（旧版は700ノードに重複）")
df_byoin[["hospital_name", "address", "市区町村ID", "二次医療圏"]].head()

一意な病院数: 87 （旧版は700ノードに重複）


,hospital_name,address,市区町村ID,二次医療圏
0,伊東市民病院,日本、〒414-0055 静岡県伊東市岡１９６−１,22208,熱海伊東
1,国際医療福祉大学熱海病院,日本、〒413-0012 静岡県熱海市東海岸町１３−１,22205,熱海伊東
2,独立行政法人国立病院機構静岡医療センター,日本、〒411-8611 静岡県駿東郡清水町長沢７６２−１,22341,駿東田方
3,沼津市立病院,日本、〒410-0302 静岡県沼津市東椎路春ノ木５５０,22203,駿東田方
4,独立行政法人地域医療機能推進機構三島総合病院,日本、〒411-0801 静岡県三島市谷田藤久保２２７６,22206,駿東田方


## 5. ノード構築（病院＋避難所）と人口配分

In [6]:
def build_nodes(df_byoin, df_hinan):
    byoin = df_byoin.copy()
    byoin["node_id"] = [f"H_{i:04d}" for i in range(len(byoin))]
    byoin["node_type"] = "HOSPITAL"
    byoin.rename(columns={"市区町村ID": "municipality_id", "二次医療圏ID": "area2_id",
                          "二次医療圏": "area2_name"}, inplace=True)
    hinan = df_hinan.copy()
    hinan["node_id"] = [f"S_{i:04d}" for i in range(len(hinan))]
    hinan["node_type"] = "SHELTER"
    hinan.rename(columns={"緯度": "latitude", "経度": "longitude", "市区町村ID": "municipality_id",
                          "二次医療圏ID": "area2_id", "二次医療圏": "area2_name"}, inplace=True)
    cols = ["node_id", "node_type", "municipality_id", "area2_id", "area2_name", "latitude", "longitude"]
    return pd.concat([byoin[cols], hinan[cols]], ignore_index=True)

def build_node_population(df_jinko, nodes):
    # 政令市の区コードを市コードへ正規化し、同一市区町村のノードへ人口を均等配分
    nodes = nodes.copy()
    nodes["muni_norm"] = nodes["municipality_id"].apply(normalize_muni)
    muni_to_nodes = nodes.groupby("muni_norm")["node_id"].apply(list).to_dict()
    records, dropped = [], 0.0
    for _, row in df_jinko.iterrows():
        muni = normalize_muni(row["市区町村ID"]); pop = row["総数"]
        node_list = muni_to_nodes.get(muni, [])
        if not node_list:
            dropped += float(pop); continue
        share = float(pop) / len(node_list)
        for nid in node_list:
            records.append({"node_id": nid, "pop": share})
    if dropped > 0:
        print(f"【注意】ノード不在で配分されなかった人口: {dropped:,.0f}")
    return pd.DataFrame(records).groupby("node_id", as_index=False)["pop"].sum()

df_jinko["総数"] = pd.to_numeric(df_jinko["総数"], errors="coerce").fillna(0)
nodes = build_nodes(df_byoin, df_hinanzyo)
nodes.to_csv("csvv3/nodes.csv", index=False)
node_pop = build_node_population(df_jinko, nodes)
node_pop.to_csv("csvv3/node_pop.csv", index=False)
print("病院:", (nodes.node_type=="HOSPITAL").sum(), " 避難所:", (nodes.node_type=="SHELTER").sum(),
      " 総ノード:", len(nodes))
print("人口付きノード:", len(node_pop), " 総人口:", f"{node_pop['pop'].sum():,.0f}")

病院: 87  避難所: 1701  総ノード: 1788
人口付きノード: 1788  総人口: 3,596,994


## 6. 薬剤（製品＝医薬品名レベル, 314製品）

In [7]:
drug_usage_df = pd.read_excel("医薬品/静岡県糖尿病薬のみ.xlsx")
drug_usage_df = drug_usage_df[drug_usage_df["薬効分類名称"] == "糖尿病用剤"].copy()
drug_usage_df = drug_usage_df[drug_usage_df["静岡県"] > 0].copy().reset_index(drop=True)
drug_usage_df["class_id"] = [f"C_{i:03d}" for i in range(len(drug_usage_df))]
drug_class_map_df = drug_usage_df[["医薬品名", "class_id"]]
drug_class_map_df.to_csv("csvv3/drug_class_map.csv", index=False)
drug_usage_df.to_csv("csvv3/drug_usage_df.csv", index=False)

# ★重要：demand は「錠数」建て（mg換算しない）。元パイプライン(csvv2)準拠。
#   モデル側 real_paramsv7 が pd_per_tab = dose_mg/RDD で患者・日へ換算するため、
#   ここで dose_mg を掛けると二重換算になり需要が桁違いに膨張する。
drug_use_for_model = (drug_usage_df.groupby("class_id", as_index=False)["静岡県"].sum()
                      .rename(columns={"静岡県": "annual_use"}))
drug_use_for_model.to_csv("csvv3/drug_use_by_class.csv", index=False)
print("製品(class_id)数:", len(drug_use_for_model), " ※annual_useは錠数建て")

製品(class_id)数: 314  ※annual_useは錠数建て


## 7. baseline_demand / demand（ノード×製品×シナリオ）

In [8]:
PERIOD_DAYS = 7.0
total_population = node_pop["pop"].sum()
node_pop = node_pop.copy()
node_pop["pop_share"] = node_pop["pop"] / total_population

# baseline（ノード×製品, mg）
duse = list(drug_use_for_model.itertuples(index=False))
records = []
for nrow in node_pop.itertuples(index=False):
    for d in duse:
        period_mg = d.annual_use / 365.0 * PERIOD_DAYS
        records.append({"node_id": nrow.node_id, "class_id": d.class_id,
                        "baseline_demand_mg": period_mg * nrow.pop_share})
baseline_demand = pd.DataFrame(records)
baseline_demand.to_csv("csvv3/baseline_demand.csv", index=False)

# シナリオ
scenarios = pd.DataFrame([
    {"scenario_id": "S1", "prob": 0.5,  "multiplier": 1.0, "shelter_multiplier": 1.0},
    {"scenario_id": "S2", "prob": 0.35, "multiplier": 1.2, "shelter_multiplier": 1.5},
    {"scenario_id": "S3", "prob": 0.15, "multiplier": 1.5, "shelter_multiplier": 2.0},
])
scenarios.to_csv("csvv3/scenarios.csv", index=False)

node_type = nodes.set_index("node_id")["node_type"].to_dict()
records = []
for b in baseline_demand.itertuples(index=False):
    ntype = node_type.get(b.node_id, "HOSPITAL")
    for s in scenarios.itertuples(index=False):
        val = b.baseline_demand_mg * s.multiplier * (s.shelter_multiplier if ntype == "SHELTER" else 1.0)
        records.append({"node_id": b.node_id, "class_id": b.class_id,
                        "scenario_id": s.scenario_id, "demand_mg": val})
demand_df = pd.DataFrame(records)
demand_df.to_csv("csvv3/demand.csv", index=False)
print("demand.csv 行数:", len(demand_df))

demand.csv 行数: 1684296


## 8. 防災倉庫

In [9]:
wh_raw = pd.read_csv("医療品等備蓄センター／赤十字血液センター(2020).csv").reset_index(drop=True)
wh_raw["warehouse_id"] = [f"W{i+1:03d}" for i in range(len(wh_raw))]
warehouses_df = wh_raw.rename(columns={"名称": "name", "所在地": "address", "施設種別": "type",
                                       "緯度": "latitude", "経度": "longitude"})[
    ["warehouse_id", "name", "address", "type", "latitude", "longitude"]]
warehouses_df.to_csv("csvv3/warehouses.csv", index=False)
print("倉庫数:", len(warehouses_df))
print("=== csvv3 生成完了 ===")

倉庫数: 16
=== csvv3 生成完了 ===
